# Detection Parameter Tuning

**Purpose:** This notebook is for tuning detection parameters on a NEW video. After tuning, edit `src/config.py` with the values you found, then run the production pipeline:

```
python -m src.detection
python -m src.tracking
python -m src.analysis
python -m src.visualize_tracks
```

The reference video this codebase was tuned on (`data/image_3.mp4`) is 1280 × 1024 at ~20 fps, ~400 frames, with ~99 Daphnia in a 165 × 145 mm side-view tank. If your video is similar, the existing defaults in `src/config.py` will be a sensible starting point. If lighting, density, framing, or animal size differ, retune `MOG2_VAR_THRESHOLD`, `MIN_AREA`, `MAX_AREA`, and the four `ROI_*` margins first.

## Setup

Imports, a `sys.path` shim so `src` is importable from this directory, and a sanity-check that prints the OpenCV version and all config values from `src/config.py`.

Run this cell first every time you open the notebook. If you edit `src/config.py` mid-session, re-run this cell to pick up the new values via `importlib.reload(cfg)`.

In [ ]:
import sys
import importlib
from pathlib import Path

# Make src importable whether Jupyter is launched from repo root or notebooks/
_here = Path().resolve()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

import src.config as cfg
importlib.reload(cfg)

print(f'OpenCV : {cv2.__version__}')
print(f'NumPy  : {np.__version__}')
print()
print('--- Config values ---')
for _name in sorted(dir(cfg)):
    if _name.isupper():
        print(f'  {_name} = {getattr(cfg, _name)!r}')

## Video Properties

Read basic metadata from the input video: resolution, FPS, frame count, and duration. These values drive all downstream index arithmetic.

In [ ]:
cap = cv2.VideoCapture(str(cfg.VIDEO_PATH))
assert cap.isOpened(), f'Cannot open {cfg.VIDEO_PATH}'

width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps      = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = n_frames / fps if fps > 0 else float('nan')

cap.release()

print(f'Resolution : {width} x {height} px')
print(f'FPS        : {fps:.2f}')
print(f'Frames     : {n_frames}')
print(f'Duration   : {duration:.2f} s')

## Sample Frames

Extract four evenly-spaced frames and display them in a 2 x 2 grid to get a first look at the video. `get_frame()` is defined here and reused throughout the notebook.

In [ ]:
def get_frame(video_path: Path, idx: int) -> np.ndarray:
    """Return frame at position idx as a grayscale uint8 array."""
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f'Could not read frame {idx} from {video_path}')
    return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame


cap = cv2.VideoCapture(str(cfg.VIDEO_PATH))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

sample_indices = [int(n_frames * f) for f in (0.1, 0.33, 0.66, 0.9)]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, idx in zip(axes.flat, sample_indices):
    frame = get_frame(cfg.VIDEO_PATH, idx)
    ax.imshow(frame, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'frame {idx}')
    ax.axis('off')
plt.suptitle('Sample frames', fontsize=13)
plt.tight_layout()
plt.show()

## Background Subtraction

Build a MOG2 background subtractor with the parameters from `src/config.py`, warm it up on the first `MOG2_WARMUP_FRAMES` frames, then apply it to a representative middle frame.

Three panels:
1. **Original frame** — raw grayscale
2. **Raw foreground mask** — straight out of MOG2
3. **Morphologically opened mask** — after suppressing small noise blobs with `cv2.MORPH_OPEN`

**Key question:** Are stationary artefacts (bubble columns, debris, persistent reflections) absorbed into the background model after the warmup period? If not, the ROI crop in a later cell can handle them. The reference video has a bubble column at the top and debris at the bottom, both excluded by `ROI_TOP` and `ROI_BOTTOM`.

In [ ]:
cap = cv2.VideoCapture(str(cfg.VIDEO_PATH))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fgbg = cv2.createBackgroundSubtractorMOG2(
    history=cfg.MOG2_HISTORY,
    varThreshold=cfg.MOG2_VAR_THRESHOLD,
    detectShadows=cfg.MOG2_DETECT_SHADOWS,
)

print(f'Warming up on {cfg.MOG2_WARMUP_FRAMES} frames...')
for i in range(cfg.MOG2_WARMUP_FRAMES):
    ok, frame = cap.read()
    if not ok:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
    fgbg.apply(gray)

# Representative middle frame (apply without updating the model)
mid_idx = n_frames // 2
cap.set(cv2.CAP_PROP_POS_FRAMES, mid_idx)
ok, frame = cap.read()
cap.release()
assert ok, 'Could not read middle frame'

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
raw_mask = fgbg.apply(gray, learningRate=0)

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (cfg.MORPH_KERNEL_SIZE, cfg.MORPH_KERNEL_SIZE)
)
opened_mask = cv2.morphologyEx(
    raw_mask, cv2.MORPH_OPEN, kernel, iterations=cfg.MORPH_OPEN_ITERATIONS
)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(gray,        cmap='gray'); axes[0].set_title('Original');              axes[0].axis('off')
axes[1].imshow(raw_mask,    cmap='gray'); axes[1].set_title('Raw MOG2 mask');         axes[1].axis('off')
axes[2].imshow(opened_mask, cmap='gray'); axes[2].set_title('After morphological open'); axes[2].axis('off')
plt.suptitle(f'Background subtraction — frame {mid_idx}', fontsize=12)
plt.tight_layout()
plt.show()
print('fgbg model built and available as `fgbg` for later cells.')

**Observations (fill in after running the cell above):**

- Stationary artefacts (bubbles, debris, reflections): absorbed into background? _yes / partially / no_
- Overall mask quality: _clean / noisy / other_
- Daphnia visible in the mask as discrete blobs? _yes / mostly / no_
- Suggested parameter adjustments (if any): typically `MOG2_VAR_THRESHOLD` (lower = more sensitive, higher = stricter)

## Contour Area Distribution

*Depends on: Background Subtraction cell (provides `opened_mask`, `mid_idx`).*

Find all contours in the opened mask and plot their area distribution on a log y-axis. The goal is to identify three populations:

- **Noise** — very small areas (isolated pixels, vibration artefacts)
- **Daphnia** — the cluster of interest
- **Clumps / debris** — large areas (touching Daphnia, stationary objects)

Use this histogram to choose `MIN_AREA` and `MAX_AREA` in `src/config.py`. The vertical dashed lines show the current config values. Note that contours above `WATERSHED_SPLIT_THRESHOLD_PX2` will be passed to the watershed splitter in production, so they need not be excluded by `MAX_AREA`.

In [ ]:
contours, _ = cv2.findContours(opened_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
areas = np.array([cv2.contourArea(c) for c in contours], dtype=float)

print(f'Total contours on frame {mid_idx}: {len(areas)}')
if len(areas):
    stats = pd.Series(areas).describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    print(stats.to_string())

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(areas, bins=60, edgecolor='k', linewidth=0.4, color='steelblue')
    ax.axvline(cfg.MIN_AREA, color='green', linestyle='--', linewidth=1.5,
               label=f'MIN_AREA = {cfg.MIN_AREA}')
    ax.axvline(cfg.MAX_AREA, color='red',   linestyle='--', linewidth=1.5,
               label=f'MAX_AREA = {cfg.MAX_AREA}')
    ax.set_yscale('log')
    ax.set_xlabel('Contour area (px²)')
    ax.set_ylabel('Count (log scale)')
    ax.set_title(f'Contour area distribution — frame {mid_idx}')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No contours found — check background subtraction settings.')

## Detection Visualization

*Depends on: Background Subtraction cell (provides `gray`, `contours`, `mid_idx`).*

Compare detections **before** and **after** area filtering on the same frame:

- **Red circles** — all contour centroids, no area filter
- **Green circles** — only contours passing `MIN_AREA ≤ area ≤ MAX_AREA`

To iterate: update `MIN_AREA` / `MAX_AREA` in `src/config.py`, then re-run the Setup cell (the `importlib.reload(cfg)` call there picks up the new values) and re-run this cell.

In [ ]:
def draw_centroids(frame_gray, contours, min_area=0.0, max_area=float('inf'), color=(50, 200, 50)):
    """Return an RGB copy of frame_gray with centroid dots drawn for qualifying contours."""
    vis = cv2.cvtColor(frame_gray, cv2.COLOR_GRAY2RGB)
    for c in contours:
        area = cv2.contourArea(c)
        if min_area <= area <= max_area:
            M = cv2.moments(c)
            if M['m00'] > 0:
                cx = int(M['m10'] / M['m00'])
                cy = int(M['m01'] / M['m00'])
                cv2.circle(vis, (cx, cy), 4, color, -1)
    return vis


n_all  = len(contours)
n_filt = sum(1 for c in contours if cfg.MIN_AREA <= cv2.contourArea(c) <= cfg.MAX_AREA)

all_vis  = draw_centroids(gray, contours, color=(220, 50, 50))
filt_vis = draw_centroids(gray, contours, cfg.MIN_AREA, cfg.MAX_AREA, color=(50, 200, 50))

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(all_vis);  axes[0].set_title(f'All contours ({n_all})');    axes[0].axis('off')
axes[1].imshow(filt_vis); axes[1].set_title(f'Area-filtered ({n_filt})'); axes[1].axis('off')
plt.suptitle(f'Detection visualization — frame {mid_idx}', fontsize=12)
plt.tight_layout()
plt.show()

**Observations:**

- Filtered detection count: ___
- False positives remaining (non-Daphnia with green circle): ___
- Missed Daphnia (visible in frame, no green circle): ___
- Does `MIN_AREA` / `MAX_AREA` need adjustment? Notes:

## ROI Mask

*Depends on: Background Subtraction cell (provides `gray`).*

Define pixel margins to exclude zones that permanently produce non-Daphnia detections: the bubble column at the top, debris at the bottom, and any wall artefacts at the sides. The ROI mask zeroes out `ROI_TOP` rows at the top, `ROI_BOTTOM` rows at the bottom, `ROI_LEFT` columns at the left, and `ROI_RIGHT` columns at the right before contour detection. The remaining region should correspond to the inside of the tank.

Excluded zones are shown as a **semi-transparent red overlay**. Adjust the variables at the top of the cell until the overlay covers the problematic areas without encroaching on Daphnia-occupied territory. Then commit the final values to `src/config.py`.

In [ ]:
# Adjust these, then copy final values to src/config.py
roi_top    = cfg.ROI_TOP
roi_bottom = cfg.ROI_BOTTOM
roi_left   = cfg.ROI_LEFT
roi_right  = cfg.ROI_RIGHT

h, w = gray.shape

roi_mask = np.zeros((h, w), dtype=np.uint8)
y0 = roi_top
y1 = h - roi_bottom if roi_bottom > 0 else h
x0 = roi_left
x1 = w - roi_right  if roi_right  > 0 else w
roi_mask[y0:y1, x0:x1] = 255

# Semi-transparent red overlay on excluded pixels
vis_roi = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB).astype(float)
excluded = (roi_mask == 0)
vis_roi[excluded] = vis_roi[excluded] * 0.35 + np.array([190, 40, 40]) * 0.65
vis_roi = vis_roi.clip(0, 255).astype(np.uint8)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(vis_roi)
ax.set_title(
    f'ROI exclusion zones  '
    f'(top={roi_top}, bottom={roi_bottom}, left={roi_left}, right={roi_right})'
)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Active detection area: {x1 - x0} x {y1 - y0} px  '
      f'(out of {w} x {h})')

**Observations:**

- Are all bubble / reflection areas covered by the top exclusion zone? _yes / no_
- Are bottom debris / stationary particles covered? _yes / no_
- Are side-wall artefacts covered? _yes / no_
- Does the ROI clip any Daphnia-occupied area that should be kept? _yes / no_
- Final values to commit: `ROI_TOP=___, ROI_BOTTOM=___, ROI_LEFT=___, ROI_RIGHT=___`

## Combined: ROI + Size-Filtered Detections

*Depends on: Background Subtraction cell (`fgbg`) and ROI Mask cell (`roi_mask`). Run both first.*

Apply both the ROI mask and the area filter together — this is the complete detection step as implemented in `src/detection.py` (modulo watershed splitting, the degenerate-ellipse filter, and the shadow filter, which the notebook does not exercise). Run on three frames at 25%, 50%, and 75% of the video to check that detection counts are consistently in the expected band for your animal density.

In [ ]:
def detect_on_frame(fgbg_model, video_path, frame_idx, roi_mask):
    """Full detection pass on a single frame (no model update).
    Returns (frame_gray, detections) where each detection is (cx, cy, area)."""
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f'Cannot read frame {frame_idx}')

    gray_ = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
    mask = fgbg_model.apply(gray_, learningRate=0)

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (cfg.MORPH_KERNEL_SIZE, cfg.MORPH_KERNEL_SIZE)
    )
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel,
                            iterations=cfg.MORPH_OPEN_ITERATIONS)
    mask = cv2.bitwise_and(mask, roi_mask)

    contours_, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detections = []
    for c in contours_:
        area = cv2.contourArea(c)
        if cfg.MIN_AREA <= area <= cfg.MAX_AREA:
            M = cv2.moments(c)
            if M['m00'] > 0:
                cx = int(M['m10'] / M['m00'])
                cy = int(M['m01'] / M['m00'])
                detections.append((cx, cy, area))
    return gray_, detections


cap_tmp = cv2.VideoCapture(str(cfg.VIDEO_PATH))
n_frames_total = int(cap_tmp.get(cv2.CAP_PROP_FRAME_COUNT))
cap_tmp.release()

check_indices = [int(n_frames_total * f) for f in (0.25, 0.50, 0.75)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, idx in zip(axes, check_indices):
    frame_gray, dets = detect_on_frame(fgbg, cfg.VIDEO_PATH, idx, roi_mask)
    vis = cv2.cvtColor(frame_gray, cv2.COLOR_GRAY2RGB)
    for cx, cy, _ in dets:
        cv2.circle(vis, (cx, cy), 4, (50, 200, 50), -1)
    ax.imshow(vis)
    ax.set_title(f'frame {idx} — {len(dets)} detections')
    ax.axis('off')

plt.suptitle('Combined ROI + area-filtered detections', fontsize=12)
plt.tight_layout()
plt.show()

print('Detection counts:')
for idx in check_indices:
    _, dets = detect_on_frame(fgbg, cfg.VIDEO_PATH, idx, roi_mask)
    print(f'  frame {idx:5d}: {len(dets):3d}')

## Detection Count Sanity Check

**Success criterion:** Detection count should land in a band slightly below your known animal count. Slightly below because some genuine Daphnia in dense moments will merge into clumps that watershed splitting cannot perfectly resolve.

For the reference video (~99 Daphnia) the expected band is **80–100 detections per frame**. For your video, set `LOW` and `HIGH` below to roughly `0.8 × known_count` and `1.05 × known_count`. If you don't have a reliable count, eyeball it from the first sample frame.

In [ ]:
LOW, HIGH = 80, 100  # acceptable detection-count band — set for your animal density

print(f'Detection count sanity check  (expected {LOW}–{HIGH} per frame)')
print(f'{"Frame":>8}  {"Count":>6}  {"Pass?"}')
print('-' * 30)

all_pass = True
for idx in check_indices:
    _, dets = detect_on_frame(fgbg, cfg.VIDEO_PATH, idx, roi_mask)
    n = len(dets)
    ok_flag = LOW <= n <= HIGH
    all_pass = all_pass and ok_flag
    status = 'PASS' if ok_flag else f'FAIL  ({n} outside {LOW}–{HIGH})'
    print(f'{idx:>8}  {n:>6}  {status}')

print()
print('Overall:', 'PASS' if all_pass else 'FAIL — tune parameters and re-run')

## Commit Tuned Values to Config

When you are satisfied with the detection results above, **manually edit `src/config.py`** to set the final values for:

- `ROI_TOP`, `ROI_BOTTOM`, `ROI_LEFT`, `ROI_RIGHT`
- `MIN_AREA`, `MAX_AREA`
- `MOG2_HISTORY`, `MOG2_VAR_THRESHOLD`, `MOG2_WARMUP_FRAMES`
- `MORPH_KERNEL_SIZE`, `MORPH_OPEN_ITERATIONS`

**Do not auto-edit `src/config.py` from this notebook.** The edit is a deliberate, reviewable change to the project's baseline parameters.

Once committed, run the production pipeline:

```
python -m src.detection
python -m src.tracking
python -m src.analysis
python -m src.visualize_tracks
```

Spot-check the resulting `output/{stem}_annotated.mp4` and the seven values in `output/{stem}_summary.csv`.